#  Install Libraries & Import Dependencies

In [1]:
# Cell 1: Install libraries and import dependencies

import subprocess
subprocess.check_call(['pip', 'install', 'scikit-learn', '--quiet'])

import boto3
import pandas as pd
import numpy as np
import os

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Confirm versions ────────────────────────────────────────────────────────
import sklearn
print(f"✅ boto3      version : {boto3.__version__}")
print(f"✅ pandas     version : {pd.__version__}")
print(f"✅ numpy      version : {np.__version__}")
print(f"✅ sklearn    version : {sklearn.__version__}")

# ── Confirm AWS region ──────────────────────────────────────────────────────
session = boto3.session.Session()
print(f"\n✅ AWS Region : {session.region_name}")

# ── S3 Details ──────────────────────────────────────────────────────────────
BUCKET = 'ali-week6-task'
PREFIX = 'ames-housing/preprocessed'
REGION = 'us-east-1'

print(f"\n📦 S3 Bucket  : {BUCKET}")
print(f"📁 S3 Prefix  : {PREFIX}")
print(f"\n✅ Setup complete! Ready to load data.")

✅ boto3      version : 1.42.47
✅ pandas     version : 2.3.3
✅ numpy      version : 1.26.4
✅ sklearn    version : 1.8.0

✅ AWS Region : us-east-1

📦 S3 Bucket  : ali-week6-task
📁 S3 Prefix  : ames-housing/preprocessed

✅ Setup complete! Ready to load data.


# Load Train & Test CSV from S3

In [2]:
# Cell 2: Load train.csv and test.csv directly from S3

s3 = boto3.client('s3', region_name=REGION)

# ── Download from S3 to SageMaker local storage ─────────────────────────────
s3.download_file(BUCKET, f'{PREFIX}/train.csv', 'train.csv')
s3.download_file(BUCKET, f'{PREFIX}/test.csv',  'test.csv')
print("✅ Downloaded train.csv and test.csv from S3")

# ── Load into DataFrames ────────────────────────────────────────────────────
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f"\n📊 Train shape : {train_df.shape}")
print(f"📊 Test  shape : {test_df.shape}")

# ── Separate features and target ────────────────────────────────────────────
X_train = train_df.drop(columns=['SalePrice_log']).values
y_train = train_df['SalePrice_log'].values

X_test  = test_df.drop(columns=['SalePrice_log']).values
y_test  = test_df['SalePrice_log'].values

print(f"\n✅ X_train shape : {X_train.shape}")
print(f"✅ y_train shape : {y_train.shape}")
print(f"✅ X_test  shape : {X_test.shape}")
print(f"✅ y_test  shape : {y_test.shape}")

print(f"\n🎯 Target (log scale) sample values:")
print(f"   y_train[:5] = {y_train[:5].round(4)}")
print(f"\n✅ Data loaded and ready for training!")

✅ Downloaded train.csv and test.csv from S3

📊 Train shape : (2344, 282)
📊 Test  shape : (586, 282)

✅ X_train shape : (2344, 281)
✅ y_train shape : (2344,)
✅ X_test  shape : (586, 281)
✅ y_test  shape : (586,)

🎯 Target (log scale) sample values:
   y_train[:5] = [11.9316 12.1281 11.5308 11.4076 11.4569]

✅ Data loaded and ready for training!


# Train MLP Model

In [3]:
# Cell 3: Build and train MLP Regressor model

import time

# ── Define MLP Model ────────────────────────────────────────────────────────
mlp = MLPRegressor(
    hidden_layer_sizes  = (256, 128, 64),  # 3 hidden layers
    activation          = 'relu',           # ReLU activation
    solver              = 'adam',           # Adam optimizer
    learning_rate_init  = 0.001,            # initial learning rate
    max_iter            = 500,              # max epochs
    early_stopping      = True,             # stop if no improvement
    validation_fraction = 0.1,              # 10% of train for validation
    n_iter_no_change    = 20,               # patience
    batch_size          = 64,               # mini batch size
    random_state        = 42,
    verbose             = False
)

print("🏗️  MLP Architecture:")
print(f"   Input Layer    : {X_train.shape[1]} features")
print(f"   Hidden Layer 1 : 256 neurons (ReLU)")
print(f"   Hidden Layer 2 : 128 neurons (ReLU)")
print(f"   Hidden Layer 3 : 64  neurons (ReLU)")
print(f"   Output Layer   : 1 neuron (SalePrice_log)")

print(f"\n⏳ Training started...")
start_time = time.time()

# ── Train ───────────────────────────────────────────────────────────────────
mlp.fit(X_train, y_train)

end_time = time.time()
training_time = round(end_time - start_time, 2)

print(f"\n✅ Training complete!")
print(f"⏱️  Training time        : {training_time} seconds")
print(f"📉 Final loss (MSE)     : {mlp.loss_:.6f}")
print(f"🔄 Actual iterations    : {mlp.n_iter_}")
print(f"📋 Loss curve points    : {len(mlp.loss_curve_)}")

🏗️  MLP Architecture:
   Input Layer    : 281 features
   Hidden Layer 1 : 256 neurons (ReLU)
   Hidden Layer 2 : 128 neurons (ReLU)
   Hidden Layer 3 : 64  neurons (ReLU)
   Output Layer   : 1 neuron (SalePrice_log)

⏳ Training started...

✅ Training complete!
⏱️  Training time        : 22.05 seconds
📉 Final loss (MSE)     : 0.002837
🔄 Actual iterations    : 83
📋 Loss curve points    : 83


# Evaluate Model

In [4]:
# Cell 4: Evaluate MLP model performance

# ── Make Predictions ────────────────────────────────────────────────────────
y_pred_log   = mlp.predict(X_test)           # predicted log values
y_pred_actual = np.expm1(y_pred_log)          # convert back to real $ values
y_test_actual = np.expm1(y_test)              # convert test target back to real $

# ── Metrics on LOG scale ────────────────────────────────────────────────────
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae_log  = mean_absolute_error(y_test, y_pred_log)
r2_log   = r2_score(y_test, y_pred_log)

# ── Metrics on ACTUAL $ scale ───────────────────────────────────────────────
rmse_actual = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae_actual  = mean_absolute_error(y_test_actual, y_pred_actual)
r2_actual   = r2_score(y_test_actual, y_pred_actual)

print("=" * 50)
print("📊 MODEL EVALUATION RESULTS")
print("=" * 50)

print(f"\n📉 Log Scale Metrics:")
print(f"   RMSE : {rmse_log:.6f}")
print(f"   MAE  : {mae_log:.6f}")
print(f"   R²   : {r2_log:.6f}")

print(f"\n💰 Actual Price Metrics:")
print(f"   RMSE : ${rmse_actual:,.2f}")
print(f"   MAE  : ${mae_actual:,.2f}")
print(f"   R²   : {r2_actual:.6f}")

print(f"\n🎯 Sample Predictions vs Actual:")
print(f"{'Actual':>12}  {'Predicted':>12}  {'Difference':>12}")
print("-" * 40)
for actual, pred in zip(y_test_actual[:8], y_pred_actual[:8]):
    diff = pred - actual
    print(f"${actual:>11,.0f}  ${pred:>11,.0f}  ${diff:>+11,.0f}")

print(f"\n✅ Evaluation complete!")

📊 MODEL EVALUATION RESULTS

📉 Log Scale Metrics:
   RMSE : 0.167217
   MAE  : 0.119537
   R²   : 0.848880

💰 Actual Price Metrics:
   RMSE : $37,790.45
   MAE  : $22,563.53
   R²   : 0.821876

🎯 Sample Predictions vs Actual:
      Actual     Predicted    Difference
----------------------------------------
$    161,000  $    161,334  $       +334
$    116,000  $    113,641  $     -2,359
$    196,500  $    154,979  $    -41,521
$    123,600  $    108,113  $    -15,487
$    126,000  $    120,763  $     -5,237
$    174,190  $    169,159  $     -5,031
$    200,000  $    153,510  $    -46,490
$    148,500  $    148,770  $       +270

✅ Evaluation complete!


# Save Predictions & Results to S3

In [5]:
# Cell 5: Save predictions and results to S3

# ── Build Results DataFrame ─────────────────────────────────────────────────
results_df = pd.DataFrame({
    'Actual_Price'     : y_test_actual,
    'Predicted_Price'  : y_pred_actual,
    'Difference'       : y_pred_actual - y_test_actual,
    'Abs_Error'        : np.abs(y_pred_actual - y_test_actual),
    'Actual_log'       : y_test,
    'Predicted_log'    : y_pred_log
})

# ── Build Metrics DataFrame ─────────────────────────────────────────────────
metrics_df = pd.DataFrame({
    'Metric'  : ['RMSE_log', 'MAE_log', 'R2_log',
                 'RMSE_actual', 'MAE_actual', 'R2_actual',
                 'Training_time_sec', 'Actual_iterations'],
    'Value'   : [rmse_log, mae_log, r2_log,
                 rmse_actual, mae_actual, r2_actual,
                 training_time, mlp.n_iter_]
})

# ── Save locally first ──────────────────────────────────────────────────────
results_df.to_csv('predictions.csv', index=False)
metrics_df.to_csv('metrics.csv',     index=False)
print("✅ Saved locally: predictions.csv & metrics.csv")

# ── Upload to S3 ────────────────────────────────────────────────────────────
RESULTS_PREFIX = 'ames-housing/results'

for filename in ['predictions.csv', 'metrics.csv']:
    s3_key = f"{RESULTS_PREFIX}/{filename}"
    s3.upload_file(filename, BUCKET, s3_key)
    print(f"✅ Uploaded → s3://{BUCKET}/{s3_key}")

print(f"\n📦 Results saved to S3:")
print(f"   s3://{BUCKET}/{RESULTS_PREFIX}/predictions.csv")
print(f"   s3://{BUCKET}/{RESULTS_PREFIX}/metrics.csv")

print(f"\n📊 Predictions Sample:")
print(results_df.head(5).to_string(index=False))

print(f"\n📋 Metrics Summary:")
print(metrics_df.to_string(index=False))

print(f"\n🎉 All done! MLP pipeline complete!")

✅ Saved locally: predictions.csv & metrics.csv
✅ Uploaded → s3://ali-week6-task/ames-housing/results/predictions.csv
✅ Uploaded → s3://ali-week6-task/ames-housing/results/metrics.csv

📦 Results saved to S3:
   s3://ali-week6-task/ames-housing/results/predictions.csv
   s3://ali-week6-task/ames-housing/results/metrics.csv

📊 Predictions Sample:
 Actual_Price  Predicted_Price    Difference    Abs_Error  Actual_log  Predicted_log
     161000.0    161334.229919    334.229919   334.229919   11.989166      11.991240
     116000.0    113640.775284  -2359.224716  2359.224716   11.661354      11.640806
     196500.0    154979.438899 -41520.561101 41520.561101   12.188423      11.951054
     123600.0    108112.666608 -15487.333392 15487.333392   11.724814      11.590938
     126000.0    120762.567552  -5237.432448  5237.432448   11.744045      11.701590

📋 Metrics Summary:
           Metric        Value
         RMSE_log     0.167217
          MAE_log     0.119537
           R2_log     0.848880
